In [2]:
import pandas as pd
from scipy.stats import kendalltau
import rbo  

# Load your CSV files
anova_chi2_df = pd.read_csv("p_values_for_base.csv")
value_dt = pd.read_csv("Booster_boosting_shap_output.csv")
value_svc = pd.read_csv("Booster_gnb_shap_output.csv")

anova_chi2_features = anova_chi2_df["Feature"].tolist()
value_features_dt = value_dt["Feature"].tolist()
value_features_svc = value_svc["Feature"].tolist()

# Prepare ranking dictionaries
anova_chi2_ranks = {f: i for i, f in enumerate(anova_chi2_features)}
value_ranks_dt = {f: i for i, f in enumerate(value_features_dt)}
value_ranks_svc = {f: i for i, f in enumerate(value_features_svc)}

# Common features for Kendall Tau between p and shap 1
common_features = list(set(anova_chi2_features) & set(value_features_dt))
ranks_1_a = [anova_chi2_ranks[f] for f in common_features]
ranks_2 = [value_ranks_dt[f] for f in common_features]

# Common features for Kendall Tau between p and shap 2
common_features = list(set(anova_chi2_features) & set(value_features_svc))
ranks_1_b = [anova_chi2_ranks[f] for f in common_features]
ranks_3 = [value_ranks_svc[f] for f in common_features]

# Compute Kendall's Tau
kendall_tau_score, _ = kendalltau(ranks_1_a, ranks_2) # kendall's tau score for p and dt
kendall_tau_score_2, _ = kendalltau(ranks_1_b, ranks_3) # kendall's tau score for p and svm rbf

# Compute RBO
rbo_score_1 = rbo.RankingSimilarity(anova_chi2_features, value_features_dt).rbo(p=0.9)
rbo_score_2 = rbo.RankingSimilarity(anova_chi2_features, value_features_svc).rbo(p=0.9)

# Compute Recall@K
def recall_at_k(list1, list2, k):
    return len(set(list1[:k]).intersection(set(list2[:k]))) / k

recall_10 = recall_at_k(anova_chi2_features, value_features_dt, 10)
recall_20 = recall_at_k(anova_chi2_features, value_features_dt, 20)
recall_30 = recall_at_k(anova_chi2_features, value_features_dt, 30)

recall_10_2 = recall_at_k(anova_chi2_features, value_features_svc, 10)
recall_20_2 = recall_at_k(anova_chi2_features, value_features_svc, 20)
recall_30_2 = recall_at_k(anova_chi2_features, value_features_svc, 30)

# Display results

print("Scores between anova-chi (p-values) and boosting shap values:")
print(f"Kendall's Tau: {kendall_tau_score:.3f}")
print(f"RBO (p=0.9): {rbo_score_1:.3f}")
print(f"Recall@10: {recall_10:.2f}")
print(f"Recall@20: {recall_20:.2f}")
print(f"Recall@30: {recall_30:.2f}")

print("Scores between anova-chi (p-values) and gnb shap values:")
print(f"Kendall's Tau: {kendall_tau_score_2:.3f}")
print(f"RBO (p=0.9): {rbo_score_2:.3f}")
print(f"Recall@10: {recall_10_2:.2f}")
print(f"Recall@20: {recall_20_2:.2f}")
print(f"Recall@30: {recall_30_2:.2f}")

Scores between anova-chi (p-values) and boosting shap values:
Kendall's Tau: 0.000
RBO (p=0.9): 0.349
Recall@10: 0.80
Recall@20: 0.40
Recall@30: 0.27
Scores between anova-chi (p-values) and gnb shap values:
Kendall's Tau: -0.429
RBO (p=0.9): 0.222
Recall@10: 0.80
Recall@20: 0.40
Recall@30: 0.27
